<a href="https://colab.research.google.com/github/davide-creator/project1/blob/main/X.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import os
import pandas as pd
import win32com.client as win32
import pyzipper
from PyPDF2 import PdfReader, PdfWriter
from openpyxl import load_workbook

###############################################################################
# 1) Funzione per leggere le variabili esterne dal file Excel
#    --------------------------------------------------------
#    Supponiamo che l’Excel abbia, ad esempio, le seguenti colonne:
#    - Colonna A: NomeSubfolderOutlook
#    - Colonna B: PasswordPDF
#    - Colonna C: PasswordZIP
#    - Colonna D: PathDiSalvataggio
#    - ...
#    E che la prima riga sia usata come intestazione (header).
###############################################################################
def read_config(excel_path):
    # Legge il file Excel come DataFrame
    df = pd.read_excel(excel_path)

    # Trasformiamo il DataFrame in un dizionario di configurazione.
    # La logica precisa dipende da come avete impostato l'Excel.
    # Qui ipotizziamo che esista una sola riga di parametri generali.
    # Se ne avete molte, dovrete gestire il mapping riga -> subfolder e così via.

    config = {
        'outlook_subfolder': str(df.loc[0, 'NomeSubfolderOutlook']),  # stringa
        'pdf_password': str(df.loc[0, 'PasswordPDF']),                # stringa
        'zip_password': str(df.loc[0, 'PasswordZIP']),                # stringa
        'save_path':    str(df.loc[0, 'PathDiSalvataggio']),          # stringa
    }

    return config

###############################################################################
# 2) Funzioni di utilità per rimuovere password e salvare i file
###############################################################################

def remove_pdf_password(protected_pdf_path, output_pdf_path, pdf_password):
    """
    Usa PyPDF2 per rimuovere la password da un PDF protetto.
    - protected_pdf_path: path del PDF protetto
    - output_pdf_path: path del PDF (senza password) che vogliamo generare
    - pdf_password: password da usare per decrittare il PDF
    """
    try:
        with open(protected_pdf_path, 'rb') as f_in:
            reader = PdfReader(f_in)
            # TENTATIVO di decifrare (se la password è Owner/User password)
            if reader.is_encrypted:
                reader.decrypt(pdf_password)

            writer = PdfWriter()
            for page in reader.pages:
                writer.add_page(page)

            with open(output_pdf_path, 'wb') as f_out:
                writer.write(f_out)

        # Se volete, potete rimuovere il file protetto iniziale:
        # os.remove(protected_pdf_path)

    except Exception as e:
        print(f"Errore nella rimozione password PDF: {e}")


def extract_zip_password(protected_zip_path, extract_folder, zip_password):
    """
    Usa pyzipper per estrarre file da uno ZIP protetto.
    - protected_zip_path: path dello ZIP protetto
    - extract_folder: cartella dove estrarre il contenuto
    - zip_password: password di protezione
    """
    try:
        with pyzipper.AESZipFile(protected_zip_path, 'r') as zip_ref:
            zip_ref.pwd = zip_password.encode('utf-8')
            zip_ref.extractall(path=extract_folder)

        # Se volete, potete rimuovere lo ZIP protetto originale:
        # os.remove(protected_zip_path)

    except Exception as e:
        print(f"Errore nell'estrazione ZIP protetto: {e}")


###############################################################################
# 3) Funzione principale per:
#    - Leggere la config
#    - Connettersi a Outlook
#    - Scaricare gli allegati dalla subfolder
#    - Identificare il tipo di file (PDF, ZIP, Excel, ecc.)
#    - Rimuovere password/estrarre i file
#    - Salvare il risultato nel path desiderato
###############################################################################
def main():
    # Sostituite con il vostro path del file Excel di configurazione
    excel_config_path = r"C:\path\to\config.xlsx"
    config = read_config(excel_config_path)

    outlook_subfolder = config['outlook_subfolder']
    pdf_password      = config['pdf_password']
    zip_password      = config['zip_password']
    save_path         = config['save_path']

    # Assicuriamoci che la cartella di salvataggio esista:
    if not os.path.exists(save_path):
        os.makedirs(save_path)

    # Connessione a Outlook
    outlook = win32.Dispatch("Outlook.Application").GetNamespace("MAPI")

    # Se la cartella che vi interessa è all'interno della vostra "Posta in arrivo" (Inbox),
    # potete navigare come segue. In base alla configurazione, potrebbe variare:
    inbox = outlook.GetDefaultFolder(6)  # 6 = Inbox
    try:
        # Subfolder con il nome specificato in Excel
        target_folder = inbox.Folders[outlook_subfolder]
    except Exception as e:
        print(f"Errore nel recuperare la subfolder '{outlook_subfolder}': {e}")
        return

    # Cicliamo tutti gli elementi (mail) nella cartella
    for item in target_folder.Items:
        # Se l'oggetto non è un’email, salta
        if item.Class != 43:  # 43 = MailItem
            continue

        # Se ci sono allegati, li processiamo
        if item.Attachments.Count > 0:
            for attach in item.Attachments:
                # Salviamo innanzitutto l'allegato su disco con il nome originale
                attachment_name = attach.FileName
                raw_attachment_path = os.path.join(save_path, attachment_name)
                attach.SaveAsFile(raw_attachment_path)

                # Verifichiamo estensione per capire se è pdf, zip, ecc.
                ext = os.path.splitext(attachment_name)[1].lower()

                if ext == '.pdf':
                    # Tenta di rimuovere la password (se pdf_password è corretto e il PDF è protetto)
                    # Creiamo un nuovo nome per il PDF deprotetto
                    output_pdf_path = os.path.join(save_path, f"DEPROTETTO_{attachment_name}")
                    remove_pdf_password(raw_attachment_path, output_pdf_path, pdf_password)

                elif ext == '.zip':
                    # Tenta di estrarre il contenuto (se zip_password è corretto)
                    # Creiamo una sottocartella dove estrarre
                    extract_folder = os.path.join(save_path, f"ESTRATTO_{os.path.splitext(attachment_name)[0]}")
                    if not os.path.exists(extract_folder):
                        os.makedirs(extract_folder)
                    extract_zip_password(raw_attachment_path, extract_folder, zip_password)

                else:
                    # Per altri tipi di file (xlsx, docx, csv, ecc.) semplicemente li lasciamo salvati
                    pass

    print("Processo completato.")


if __name__ == "__main__":
    main()